# 05 — Submission Phase: real hardware

The only change from notebook 02 is the config file. Before running:

1. `pip install "qlkit[iqm]"` (installs `qiskit-iqm`)
2. Set the `IQM_TOKEN` environment variable (never paste tokens into notebooks)
3. Put your assigned QC URL into `configs/iqm.json`

What the middleware does for you on hardware:
- compiles your QUBO to a QAOA circuit in IQM's native gates,
- **batches** concurrent requests into shared jobs (shared queue, 30 teams — be kind),
- polls with exponential backoff and persists job IDs to `.qlkit/jobs.json`, so a kernel restart never orphans a QPU job,
- normalizes, decodes, repairs, and scores exactly as locally.

⛔ `SolverConfig.max_qpu_jobs` (default 50) is a client-side circuit breaker. Do **not** point your overnight Bayesian loop at the QPU.

In [ ]:
import sys; sys.path.insert(0, "../..")
from problems.vrp import VehicleRoutingProblem
from qlkit import LogisticsSolver, GreedyAssignmentRepair

problem = VehicleRoutingProblem.small_instance()

# THE one-line backend switch:
solver = LogisticsSolver.from_config(problem, "../../configs/iqm.json",
                                     repair=GreedyAssignmentRepair())

result = solver.solve(lambdas={"one_hot": 12.0, "capacity": 2.0})
print(f"feasible: {result.samples.feasible_fraction:.1%}  "
      f"repaired: {result.samples.repaired_fraction:.1%}")
print(solver.validate(result.best_solution))

## Expect noise — that's the point

Feasible fraction will be *far* lower than on the simulator. Your score comes from how well your **repair heuristic** and **lambda tuning** compensate. Compare `repaired_fraction` here against notebook 02: that gap is your classical layer earning its keep.